# MCTS for the JSSP: the study on AWS SageMaker

Open this notebook **from the `mcts_jssp` folder** (next to `pyproject.toml`). Use a kernel with **Python 3.10 or newer** (for example `conda_python3` on a notebook instance, or the SageMaker Distribution image in Studio).

Run the cells **in order, one stage at a time**:

| Stage | Cell | Time on ml.c6i.12xlarge (approx.) |
|---|---|---|
| 0 | Setup (every session) | 1 min |
| 1 | Pilot study (install, tests, checks, pilot) | 15-30 min |
| 2 | Preparation: training (self-labelling), checks, self-play comparison, temperatures, networks for disturbances | 4-5 h |
| 3 | Experiments (two sessions of at most 7.5 h) | 10-12 h |
| 4 | Analysis | under 1 h |
| 5 | Download the results | 1 min |

Stages 2-4 run **in the background**: they keep running when the browser is closed. Check on them with the *Progress* cell. Start a stage only when the previous one has finished.

## 0. Setup (every session)
Defines `start`, `show` and `running`. In a later session it also installs the packages again if a restart of the space has removed them.

In [ ]:
import os, subprocess, sys, shutil
print("folder:", os.getcwd())
assert os.path.exists("pyproject.toml"), "open this notebook from the mcts_jssp folder"
assert sys.version_info >= (3, 10), f"Python {sys.version.split()[0]}: choose a kernel with Python 3.10+"
os.makedirs("logs", exist_ok=True)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# a restarted SageMaker space can lose installed packages: install them again if they are missing (a few minutes)
import importlib.util
missing = [m for m in ("torch", "torch_geometric") if importlib.util.find_spec(m) is None]
if missing and os.path.exists("logs/pilot.log"):        # before stage 1, stage 1 installs them
    print("installing", ", ".join(missing), "...")
    pip = [sys.executable, "-m", "pip", "install", "-q"]
    subprocess.run(pip + ["torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run(pip + ["torch_geometric"], check=True)
    subprocess.run(pip + ["-e", ".[test]"], check=True)

def start(steps, log, hours=None):
    """Start run_pipeline.py in the background; it keeps running when the browser is closed."""
    if running(quiet=True):
        print("a stage is still running - wait until it has finished (see the Progress cell)")
        return
    cmd = [sys.executable, "run_pipeline.py", "--steps", steps] + (["--hours", str(hours)] if hours else [])
    p = subprocess.Popen(cmd, stdout=open(f"logs/{log}.log", "a"), stderr=subprocess.STDOUT,
                         env=dict(os.environ), start_new_session=True)
    print(f"started: {steps}  (process {p.pid}, log logs/{log}.log)")

def show(log, lines=25):
    path = f"logs/{log}.log"
    print("".join(open(path).readlines()[-lines:]) if os.path.exists(path) else f"no {path} yet")

def running(quiet=False):
    out = subprocess.run(["pgrep", "-af", "run_pipeline.py|sagemaker_pilot.py"], capture_output=True, text=True).stdout.strip()
    if not quiet:
        print(out or "nothing running")
    return bool(out)

print("ready")

## Everything in one go (instead of stages 1-4)
**Not with the 8-hour limit of SageMaker Studio**: it runs the experiments without stopping and would be cut off. Use stages 1-4 instead.

Runs the pilot and then the whole study **in the background** (about 13-20 h on ml.c5.12xlarge): install, tests, environment checks, pilot, training, checks, self-play comparison, temperatures, experiments, analysis, and finally `study_results.zip`, `study_artifacts.zip` and `study_logs.zip`. Follow it with `show("all")`.

Note: nothing stops between the stages, so the networks are not checked before the experiments start. Look at `results/training/training_summary.csv` when training has finished (`show("all")` shows when); if a network has not learned, stop the run with the *Stop* cell.

In [ ]:
if not running(quiet=True):
    p = subprocess.Popen([sys.executable, "sagemaker_pilot.py", "--all"], stdout=open("logs/all.log", "a"),
                         stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
    print(f"started everything (process {p.pid}); follow it with show('all')")
else:
    print("a stage is still running")

In [ ]:
show("all", 30)

## Stop
Stops whatever is running. Finished work is kept; starting the same stage again continues where it stopped.

In [ ]:
for pattern in ("run_pipeline.py", "sagemaker_pilot.py", "-m mctsjssp", "multiprocessing.spawn"):
    subprocess.run(["pkill", "-9", "-f", "--", pattern])   # the pipeline, its commands and their workers
running()

## 1. Pilot study
Installs the packages, runs the quick tests, the environment checks and the pilot study, and writes `pilot_results.zip`. It also writes the number of workers (one per physical core: 24 on ml.c6i.12xlarge) into `study.yaml`. **Download `pilot_results.zip` and send it back before going on.**

In [ ]:
exec(open("sagemaker_pilot.py").read())

## 2. Preparation: training (self-labelling, and self-play for the comparison), checks, temperatures (background)
When it has finished, check `results/training/training_summary.csv` and `training_curves.png`: LR and fast LR must beat uniform random dispatching (`learned` = True), otherwise the pipeline stops; AZ and fast AZ are reported. The comparison of self-labelling with self-play is in `results/training/compare_training.md` and `compare_training.png`.

The last step, `disturbed`, trains the networks for disturbed processing times (RQ5) twice, fine-tuned from the nominal networks and from scratch, validates them under disturbances and writes `results/training/robustness.md`, `disturbed_over_time.png` and `disturbed_distribution.png`. On an installation whose stage 2 ran without it: `start("disturbed", "disturbed")`.


In [ ]:
start("train,check,self_play,compare_training,select_entropy,disturbed", "prepare")

## 3. Experiments (background)
Stops cleanly after 7.5 hours (`session_hours` in `study.yaml`), before the 8-hour limit of the IDE. In the next session run the Setup cell and this cell again: it continues where it stopped. The experiments run are listed under `pipeline.experiments` in `study.yaml`.

In [ ]:
start("run", "experiments")

## 4. Analysis (background)

In [ ]:
start("fixed_states,analyse,compare", "analysis")

## Progress
Run at any time. Change the log name to `prepare`, `experiments` or `analysis`.

In [ ]:
running()
show("prepare")        # or show("experiments") / show("analysis")

In [ ]:
# training summary (after stage 2)
import glob, zipfile
import pandas as pd
f = "results/training/training_summary.csv"
display(pd.read_csv(f)) if os.path.exists(f) else print("no training summary yet")
# to send back: pilot, checks, training, self-play comparison, networks for disturbances, logs and network summaries
files = (glob.glob("results/pilot/*") + glob.glob("results/validation/*") + glob.glob("results/training/**/*", recursive=True)
         + glob.glob("logs/*.log") + glob.glob("artifacts/*.json") + glob.glob("artifacts/models/*/*.json")
         + glob.glob("artifacts/scratch/models/*/*.json") + glob.glob("artifacts/self_play/models/*/*.json"))
files = sorted(f for f in set(files) if os.path.isfile(f))
with zipfile.ZipFile("preparation_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f)
print(f"preparation_results.zip: {len(files)} files ({os.path.getsize('preparation_results.zip') / 1e6:.1f} MB)")

## 5. Download the results
Then download the zip files from the file browser (right-click → Download). **Stop the instance afterwards**: it is billed for every hour it runs.

In [ ]:
for name, folder in [("study_results", "results"), ("study_artifacts", "artifacts"), ("study_logs", "logs")]:
    if os.path.exists(folder):
        shutil.make_archive(name, "zip", ".", folder)
        print(f"{name}.zip  ({os.path.getsize(name + '.zip') / 1e6:.1f} MB)")